In [5]:
import cv2
import os
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [9]:
def norme(a,b):
    return ((a.x-b.x)**2+(a.y-b.y)**2)**0.5 

def calcul_barycentre(landmarks):
    centre=[0,0]
    n=len(landmarks)
    for p in landmarks:
        centre[0]+=p.x/n
        centre[1]+=p.y/n
    return centre
    


def doigt_est_plie(tip, pip, wrist):
    
    dist_tip_poignet = norme(tip, wrist)
    dist_pip_poignet = norme(pip, wrist)
    
    
    return dist_tip_poignet < dist_pip_poignet

def est_poing_ferme(landmarks):
    poignet = landmarks[0]
    
    # On vérifie les 4 doigts principaux (on exclut souvent le pouce 
    # car sa mécanique de pliure est différente)
    index_plie = doigt_est_plie(landmarks[8], landmarks[6], poignet)
    majeur_plie = doigt_est_plie(landmarks[12], landmarks[10], poignet)
    annulaire_plie = doigt_est_plie(landmarks[16], landmarks[14], poignet)
    auriculaire_plie = doigt_est_plie(landmarks[20], landmarks[18], poignet)
    
    # Si les 4 doigts sont pliés vers le poignet, c'est un poing !
    if index_plie and majeur_plie and annulaire_plie and auriculaire_plie:
        return True
    return False

def est_corne(landmarks):
    poignet = landmarks[0]
    
    index_plie = doigt_est_plie(landmarks[8], landmarks[6], poignet)
    majeur_plie = doigt_est_plie(landmarks[12], landmarks[10], poignet)
    annulaire_plie = doigt_est_plie(landmarks[16], landmarks[14], poignet)
    auriculaire_plie = doigt_est_plie(landmarks[20], landmarks[18], poignet)
    if not index_plie and majeur_plie and annulaire_plie and not auriculaire_plie:
        return True
    else : 
        return False
    
def deep_three(landmarks):
    poignet = landmarks[0]

    majeur_plie = doigt_est_plie(landmarks[12], landmarks[10], poignet)
    annulaire_plie = doigt_est_plie(landmarks[16], landmarks[14], poignet)
    auriculaire_plie = doigt_est_plie(landmarks[20], landmarks[18], poignet)

    d= norme(landmarks[4],landmarks[8])<0.1
    if d and not majeur_plie and not annulaire_plie and not auriculaire_plie:
        return True
    return False

def est_chill(landmarks):
    poignet = landmarks[0]

    majeur_plie = doigt_est_plie(landmarks[12], landmarks[10], poignet)
    annulaire_plie = doigt_est_plie(landmarks[16], landmarks[14], poignet)
    auriculaire_plie = doigt_est_plie(landmarks[20], landmarks[18], poignet)
    index_plie = doigt_est_plie(landmarks[8], landmarks[6], poignet)
    if not index_plie and not majeur_plie and annulaire_plie and auriculaire_plie:
        return True
    else : 
        return False





In [7]:
def zoom(landmarks):
    aire = norme(landmarks[17],landmarks[5])*norme(landmarks[5],landmarks[0])
    min = 0.006
    max = 0.09
    inf = 0.009
    sup =0.02
    if inf<=aire <=sup :
        vitesse = 0
    elif sup<aire<=max:
        vitesse = 7/(max-sup)*(aire-sup)+1
    elif max<aire:
        vitesse = 8
    elif aire<min:
        vitesse=-8
    else :
        vitesse = 7/(inf-min)*(aire-min)-8
    return vitesse

def tourni(landmarks): #ici on renvoie la vitesse angualaire de la caméra en fonction de la position de la main (on a une vitesse pour x et y)
    centre = calcul_barycentre(landmarks)
    en_y = 0.06*round(centre[0]*50)/50 -0.03
    en_x = 0.06*round(centre[1]*50)/50 -0.03
    return (round(en_y,3),(-1)*round(en_x,3))





In [10]:


# --- 1. TÉLÉCHARGEMENT AUTOMATIQUE DU MODÈLE ---
# URL officielle de Google pour le modèle Hand Landmark le plus récent
URL_MODELE = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
CHEMIN_MODELE = "hand_landmarker.task"

# Si le fichier n'existe pas dans le dossier, on le télécharge
if not os.path.exists(CHEMIN_MODELE):
    print("Téléchargement du modèle MediaPipe (quelques Mo)... ", end="", flush=True)
    urllib.request.urlretrieve(URL_MODELE, CHEMIN_MODELE)
    print("Terminé !")
else:
    print("Modèle déjà présent localement.")

# --- 2. CONFIGURATION DE LA NOUVELLE API (Tasks) ---
# On indique le chemin du fichier .task fraîchement téléchargé
options_base = python.BaseOptions(model_asset_path=CHEMIN_MODELE)

options = vision.HandLandmarkerOptions(
    base_options=options_base,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

# Création du détecteur
detecteur = vision.HandLandmarker.create_from_options(options)

# --- 3. BOUCLE WEBCAM ---
cap = cv2.VideoCapture(0)
print("Caméra activée. Appuyez sur 'q' pour quitter.")

while cap.isOpened():
    succes, frame = cap.read()
    if not succes:
        continue

    frame = cv2.flip(frame, 1) # Effet miroir
    hauteur, largeur = frame.shape[:2]
    
    # La nouvelle API exige un objet "mp.Image" spécifique
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    # --- 4. INFÉRENCE ---
    resultats = detecteur.detect(mp_image)

    # --- 5. EXPLOITATION ET DESSIN ---
    if resultats.hand_landmarks:
        # On récupère les points de la première main (0)
        points = resultats.hand_landmarks[0]
        
        # Dessin des 21 points
        for i, point in enumerate(points):
            x_pixel = int(point.x * largeur)
            y_pixel = int(point.y * hauteur)
            
            # Poignet (0) et Centre (9) en rouge, le reste en vert
            couleur = (0, 0, 255) if i in [0, 9] else (0, 255, 0)
            cv2.circle(frame, (x_pixel, y_pixel), 5, couleur, -1)

        
        cv2.putText(frame, f"theta :,{tourni(points)} ", (int(largeur*0.3), int(hauteur*0.5)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 0, 100), 2)
            
        # Vous avez accès aux points avec point.x et point.y (entre 0.0 et 1.0)
        # Ex: centre_x = points[9].x
        if est_poing_ferme(points):
                cv2.putText(frame, f"one punchh,{zoom(points)} ", (int(largeur*0.6), int(hauteur*0.25)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
            
        if est_corne(points):
            cv2.putText(frame, "corne (tah le basket)", (int(largeur*0.6),int(hauteur*0.25)-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        if deep_three(points):
            cv2.putText(frame, "3pnts", (int(largeur*0.6),int(hauteur*0.25)-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            
        if est_chill(points):
            cv2.putText(frame, "chill guy", (int(largeur*0.6),int(hauteur*0.25)-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("MediaPipe Tasks API - Auto Download", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Modèle déjà présent localement.
Caméra activée. Appuyez sur 'q' pour quitter.


c:\miniconda\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
